# 00 — Sintetik dataset generatori (Colab T4)

**Nima uchun sintetik ma'lumot:** real pasport rasmlarini yig'ish amaliy jihatdan imkonsiz va huquqiy risk. Sintetik dataset ikki muammoni bir vaqtda hal qiladi:

1. Model o'rgatish uchun annotatsiyalangan ma'lumot beradi (bbox va matn allaqachon ma'lum — qo'lda belgilash shart emas)
2. **Bepul Gemini kalitlarini xavfsiz ishlatish imkonini beradi.** Bepul tier'ga yuborilgan ma'lumot model o'rgatish uchun ishlatilishi mumkin, shuning uchun real pasport u yerga ketmasligi kerak. Sintetik ma'lumotda esa bunday cheklov yo'q — prompt'ni cheksiz sozlashingiz mumkin.

⚠️ **Real hujjat dizaynini piksel-aniq ko'chirmang.** O'xshash tuzilishdagi maket yetarli; aniq nusxa huquqiy muammo.


In [ ]:
!pip -q install albumentations faker pillow numpy opencv-python-headless
from google.colab import drive
drive.mount('/content/drive')

import os
OUT = '/content/drive/MyDrive/ocr-docs/synthetic'
os.makedirs(OUT, exist_ok=True)
print('output ->', OUT)


## 1. O'zbek ma'lumot generatori

MRZ **to'g'ri check digit bilan** generatsiya qilinadi — `packages/schema/validators` dagi aynan shu funksiya ishlatiladi. Bu muhim: agar sintetik MRZ check digit'i noto'g'ri bo'lsa, model o'rgatilgan ma'lumot real hujjatga mos kelmaydi.


In [ ]:
import random, sys
sys.path.insert(0, '/content/ocr-docs')  # repo ni Colab ga clone qiling
from packages.schema.validators import check_digit
from packages.schema.translit import to_mrz_name

SURNAMES = ['ALIYEV','KARIMOV','RAHMONOV','YUSUPOV','TOSHMATOV','QODIROV',
            'SAIDOV','NAZAROV','ISMOILOV','XOLMATOV','ERGASHEV','MIRZAYEV']
GIVEN_M  = ['SHOHRUH','AKMAL','JAVOHIR','BEKZOD','OTABEK','SARDOR','AZIZ']
GIVEN_F  = ['MADINA','NILUFAR','GULNORA','ZARINA','SEVARA','DILNOZA']
REGIONS  = ['10','11','12','14','16','18','20','22','24','26','28','30','32','33']

def make_pinfl(sex, birth):
    marker = '5' if sex == 'M' else '6'  # 2000-yillar
    if birth.year < 2000: marker = '3' if sex == 'M' else '4'
    body = f"{birth.day:02d}{birth.month:02d}{birth.year%100:02d}"
    return marker + body + random.choice(REGIONS) + f"{random.randint(0,9999):04d}"[:4] + str(random.randint(0,9))

def make_td1(doc_no, birth, expiry, sex, pinfl, surname, given):
    """3x30 TD1 MRZ, barcha check digit'lar to'g'ri hisoblangan."""
    b = f"{birth.year%100:02d}{birth.month:02d}{birth.day:02d}"
    e = f"{expiry.year%100:02d}{expiry.month:02d}{expiry.day:02d}"
    l1 = ('ID' + 'UZB' + doc_no + str(check_digit(doc_no)) + pinfl).ljust(30, '<')[:30]
    l2_head = b + str(check_digit(b)) + sex + e + str(check_digit(e)) + 'UZB'
    l2 = l2_head.ljust(29, '<')[:29]
    composite = l1[5:30] + l2[0:7] + l2[8:15] + l2[18:29]
    l2 += str(check_digit(composite))
    l3 = to_mrz_name(surname, given).ljust(30, '<')[:30]
    return [l1, l2, l3]


## 2. Karta maketini chizish va augmentatsiya

Augmentatsiya real telefon suratining buzilishlarini taqlid qiladi: perspektiva, soya, yorqin dog', motion blur, JPEG artefakti, qog'oz teksturasi, moiré (ekrandan surat).


In [ ]:
import cv2, numpy as np, albumentations as A
from datetime import date, timedelta

AUG = A.Compose([
    A.Perspective(scale=(0.02, 0.10), p=0.8),
    A.Rotate(limit=12, border_mode=cv2.BORDER_REPLICATE, p=0.8),
    A.RandomBrightnessContrast(0.3, 0.3, p=0.8),
    A.RandomShadow(p=0.4),
    A.MotionBlur(blur_limit=7, p=0.3),
    A.GaussNoise(p=0.4),
    A.ImageCompression(quality_lower=30, quality_upper=95, p=0.9),
])

def render_card(rec, w=1012, h=638):
    card = np.full((h, w, 3), 232, np.uint8)
    cv2.rectangle(card, (0,0), (w-1,h-1), (120,120,120), 3)
    rows = [('Familiyasi', rec['surname']), ('Ismi', rec['given']),
            ('Tugilgan sanasi', rec['birth'].strftime('%d.%m.%Y')),
            ('JSHSHIR', rec['pinfl']), ('Seriya', rec['doc_no']),
            ('Jinsi', rec['sex'])]
    boxes = {}
    for i, (label, value) in enumerate(rows):
        y = 90 + i*62
        cv2.putText(card, label+':', (40, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (90,90,90), 1)
        cv2.putText(card, str(value), (330, y), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (20,20,20), 2)
        boxes[label] = [330, y-24, 330+len(str(value))*17, y+8]
    # MRZ — pastda, OCR-B o'rniga monospace (real loyihada OCR-B shrifti qo'ying)
    for j, line in enumerate(rec['mrz']):
        cv2.putText(card, line, (30, h-90+j*30), cv2.FONT_HERSHEY_PLAIN, 1.6, (10,10,10), 2)
    boxes['mrz_zone'] = [25, h-115, w-25, h-10]
    return card, boxes

def make_record():
    sex = random.choice('MF')
    birth = date(random.randint(1960,2006), random.randint(1,12), random.randint(1,28))
    expiry = birth + timedelta(days=365*random.randint(30,55))
    surname = random.choice(SURNAMES)
    given = random.choice(GIVEN_M if sex=='M' else GIVEN_F)
    doc_no = ''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ',k=2)) + f"{random.randint(0,9999999):07d}"
    pinfl = make_pinfl(sex, birth)
    rec = dict(sex=sex, birth=birth, expiry=expiry, surname=surname,
               given=given, doc_no=doc_no, pinfl=pinfl)
    rec['mrz'] = make_td1(doc_no, birth, expiry, sex, pinfl, surname, given)
    return rec

# Sanity check: generatsiya qilingan MRZ o'z validatorimizdan o'tishi SHART
from packages.schema.validators import validate_td1
r = make_record()
v = validate_td1(r['mrz'])
print('MRZ valid:', v.ok, '| errors:', v.errors)
assert v.ok, 'Sintetik MRZ noto\'g\'ri — checksum kodini tekshiring'


## 3. Datasetni generatsiya qilish

Colab sessiyasi uzilishi normal hodisa — har 1000 rasmdan keyin checkpoint yoziladi.


In [ ]:
import json, glob
from pathlib import Path

N = 20000
img_dir = Path(OUT)/'images'; lbl_dir = Path(OUT)/'labels'
img_dir.mkdir(parents=True, exist_ok=True); lbl_dir.mkdir(parents=True, exist_ok=True)

CLASSES = ['Familiyasi','Ismi','Tugilgan sanasi','JSHSHIR','Seriya','Jinsi','mrz_zone']
start = len(glob.glob(str(img_dir/'*.jpg')))  # resume
print('resuming from', start)

meta_path = Path(OUT)/'metadata.jsonl'
with open(meta_path, 'a') as meta:
    for i in range(start, N):
        rec = make_record()
        card, boxes = render_card(rec)
        out = AUG(image=card)['image']
        name = f'{i:06d}'
        cv2.imwrite(str(img_dir/f'{name}.jpg'), out, [cv2.IMWRITE_JPEG_QUALITY, 92])
        h, w = out.shape[:2]
        with open(lbl_dir/f'{name}.txt','w') as f:
            for cls, b in boxes.items():
                cx, cy = (b[0]+b[2])/2/w, (b[1]+b[3])/2/h
                bw, bh = (b[2]-b[0])/w, (b[3]-b[1])/h
                f.write(f'{CLASSES.index(cls)} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n')
        meta.write(json.dumps({'file': f'{name}.jpg', 'pinfl': rec['pinfl'],
            'doc_no': rec['doc_no'], 'birth': rec['birth'].isoformat(),
            'surname': rec['surname'], 'given': rec['given'],
            'sex': rec['sex'], 'mrz': rec['mrz']})+'\n')
        if i % 1000 == 0: print(i, flush=True)
print('done')


## 4. Keyingi qadam

Bu dataset ikki narsa uchun ishlatiladi:

| Maqsad | Notebook | Kerakmi? |
|---|---|---|
| **Prompt tuning (bepul Gemini kalitlari bilan)** | `02_prompt_tuning.ipynb` | ✅ Ha, birinchi |
| Maydon detektori (YOLO11n) | `01_field_detector.ipynb` | 🟡 Faqat eval talab qilsa |

**Tartib muhim.** Avval prompt'ni sintetik ma'lumotda sozlang — bu bepul va real PII bilan tajriba qilish xavfi yo'q. Model o'rgatish faqat baholash natijasi talab qilganda: LLM qatlami maydon xaritalashni bajaradi, shuning uchun YOLO detektori MVP uchun shart emas.
